# Phase 6 · Submission S2 — Architecture B (Encoder–Decoder Transformer) + LoRA, chunk-level abstractive summarization

**Task:** abstractive summarization of long legal judgments. **Primary metric:** BERTScore F1 (secondary ROUGE-1 F1, ROUGE-L F1).

**Method** (same data pipeline as S1; different Phase-1 model and adaptation technique)
1. **Chunks and aligned targets** as in S1: sentence-aligned chunks — here ≤ 1,000 tokens because B's encoder reads at most 1,024 — and every reference-summary sentence assigned to the chunk holding its best-matching document sentence.
2. **Model:** the complete Phase-1 Architecture B (6-layer bidirectional encoder + 6-layer causal decoder with cross-attention, 139.3M parameters, tied output layer + output bias), loaded strictly. The encoder reads `[BOS] chunk [EOS]` (padding masked); the decoder starts from `[BOS]` and writes the chunk's part of the summary, exactly the sequence-to-sequence set-up B was pretrained with.
3. **Adaptation — LoRA (r = 8, α = 16):** on every linear projection of encoder and decoder (self-attention qkv/out, cross-attention q/k/v/out, both feed-forward layers); `B = 0` at start, so training begins exactly at Phase 1; everything else frozen; dropout off.
4. **Generation:** encoder once per chunk; decoder with cached self-attention keys/values and precomputed cross-attention keys/values; greedy, repeated 4-grams blocked, min/max length proportional to the chunk; chunk summaries joined in document order with repeated sentences removed.
5. **Validation:** the same 150 held-out training documents as S1 (same seed): validation loss for epoch selection, ROUGE-1/L of full summaries, keep-fraction length control, all chunks vs first chunk only.

**Before running (Kaggle)**
- Accelerator: **GPU T4 ×2** · Add-ons → Secrets: **`HF`** · Internet on.
- Add Input → **`rohitbatra2004/arch-data-1`** (Architecture B) and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 3–4 hours. Checkpoints every 1,000 steps and every epoch; prepared data cached; to resume, add this notebook's previous output as an input and run again.

Outputs: `/kaggle/working/P6_S2_B_EncDec_LoRA/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib, re
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P6_S2_B_EncDec_LoRA"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Chunk-level summarization data.
CHUNK_TOKENS = 1000                 # document tokens per chunk (whole sentences); B's encoder reads ≤ 1,024 tokens
MAX_TARGET_TOKENS = 640             # partial-summary tokens per chunk used in training (longer targets are cut)
MAX_CHUNKS_PER_DOC = 40             # chunks kept per document (> 99.5% of documents complete)
VAL_DOCS = 150                      # held-out training documents for validation (never trained on)

# LoRA on every linear projection of the encoder and the decoder.
LORA_RANK = 8
LORA_ALPHA = 16.0

# Training.
EPOCHS = 2
BATCH_SEQS = 8                      # (chunk, summary) pairs per forward pass, split over both GPUs
GRAD_ACCUM = 2                      # effective batch = 16 pairs
LR = 3e-4                           # LoRA starts at B = 0
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.03
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 1000

# Generation (greedy, cached).
GEN_BATCH = 16
MIN_NEW_FRACTION = 0.08             # minimum summary tokens per chunk = 8% of the chunk's tokens …
MAX_NEW_FRACTION = 0.45             # … maximum = 45% of the chunk's tokens (+ 32), capped at MAX_TARGET_TOKENS
NO_REPEAT_NGRAM = 4
KEEP_FRACTIONS = (0.5, 0.7, 0.85, 1.0)   # share of each chunk summary's sentences kept (tuned on validation)
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download Phase_6_Summ from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_6_Summ/**"])
P6_DIR = DATA_DIR / "Phase_6_Summ"
for name in ("Train/train.json", "Test/test.json"):
    assert (P6_DIR / name).is_file(), f"Missing {name}"
print("Phase 6 data ready in", P6_DIR)

In [ ]:
# Cell 4 — Load train / test, drop empty summaries and duplicate documents, and hold out VAL_DOCS training documents.
def clean(text):
    return " ".join(text.split())

raw_train = json.load(open(P6_DIR / "Train/train.json"))
raw_test = json.load(open(P6_DIR / "Test/test.json"))
docs, seen = [], set()
for d in raw_train:
    document, summary = clean(d["document"]), clean(d["summary"])
    key = hashlib.md5(document.encode()).hexdigest()
    if not summary or key in seen:
        continue
    seen.add(key)
    docs.append({"id": str(d["id"]), "document": document, "summary": summary})
order = list(range(len(docs)))
random.Random(SEED).shuffle(order)
val_docs = [docs[i] for i in order[:VAL_DOCS]]
train_docs = [docs[i] for i in order[VAL_DOCS:]]
test_docs = [{"id": str(d["id"]), "document": clean(d["document"])} for d in raw_test]
del raw_train, raw_test
print(f"train {len(train_docs):,} documents | validation {len(val_docs)} (held out) | test {len(test_docs)}")

In [ ]:
# Cell 5 — Sentence-aligned chunks and (chunk → partial summary) pairs: each summary sentence goes to the chunk holding its
# best-matching document sentence (highest word recall). Cached in data_cache.pt (also used by S4).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]", "[UNK]"))
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3)
SENT_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9(\"'])")
WORD = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    return [s for s in SENT_SPLIT.split(text) if s.strip()]

def encode_sentences(sentences):
    return [np.array(e.ids, dtype=np.uint16) for e in tokenizer.encode_batch([" " + s for s in sentences])] if sentences else []

def make_chunks(doc_text):
    """Chunks of whole sentences (≤ CHUNK_TOKENS tokens) and, for every document sentence, the chunk it lies in."""
    sentences = split_sentences(doc_text)
    chunks, sentence_chunk, current = [], [], []
    for ids in encode_sentences(sentences):
        pieces = [ids[i:i + CHUNK_TOKENS] for i in range(0, max(len(ids), 1), CHUNK_TOKENS)]
        sentence_chunk.append(None)
        for piece in pieces:
            if current and sum(len(p) for p in current) + len(piece) > CHUNK_TOKENS:
                chunks.append(np.concatenate(current)); current = []
            if sentence_chunk[-1] is None:
                sentence_chunk[-1] = len(chunks)
            current.append(piece)
    if current:
        chunks.append(np.concatenate(current))
    return sentences, chunks[:MAX_CHUNKS_PER_DOC], sentence_chunk

def aligned_targets(doc_sentences, sentence_chunk, n_chunks, summary):
    """Partial summary (token array) per chunk and the word recall of every alignment."""
    doc_sets = [set(WORD.findall(s.lower())) for s in doc_sentences]
    parts, recalls = [[] for _ in range(n_chunks)], []
    summary_sentences = split_sentences(summary)
    for sent, ids in zip(summary_sentences, encode_sentences(summary_sentences)):
        words = set(WORD.findall(sent.lower()))
        if not words or not doc_sets:
            continue
        scores = [len(words & ds) / len(words) for ds in doc_sets]
        best = int(np.argmax(scores))
        recalls.append(scores[best])
        chunk = min(sentence_chunk[best], n_chunks - 1)
        parts[chunk].append(ids)
    return [np.concatenate(p)[:MAX_TARGET_TOKENS] if p else np.zeros(0, dtype=np.uint16) for p in parts], recalls

def prepare(doc_list, with_targets):
    out, recalls = [], []
    for d in doc_list:
        sentences, chunks, sentence_chunk = make_chunks(d["document"])
        if not chunks:
            chunks, sentence_chunk = [np.array(tokenizer.encode(" " + d["document"]).ids[:CHUNK_TOKENS], dtype=np.uint16)], [0]
        item = {"id": d["id"], "chunks": chunks}
        if with_targets:
            item["targets"], r = aligned_targets(sentences, sentence_chunk, len(chunks), d["summary"])
            item["summary"] = d["summary"]; recalls += r
        out.append(item)
    return out, recalls

cache = WORK_DIR / "data_cache.pt"
if not cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/data_cache.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], cache)
if cache.exists():
    data = torch.load(cache, weights_only=False)
    print("Loaded prepared data from", cache)
else:
    start_time = time.time()
    train_items, recalls = prepare(train_docs, True)
    val_items, _ = prepare(val_docs, True)
    test_items, _ = prepare(test_docs, False)
    data = {"train": train_items, "val": val_items, "test": test_items, "alignment_recall": recalls,
            "settings": {"chunk_tokens": CHUNK_TOKENS, "max_target_tokens": MAX_TARGET_TOKENS}}
    torch.save(data, cache)
    print(f"Prepared chunks and aligned targets in {(time.time() - start_time) / 60:.1f} min → {cache}")
assert [d["id"] for d in data["val"]] == [d["id"] for d in val_docs] and [d["id"] for d in data["test"]] == [d["id"] for d in test_docs]

def pairs_of(items):
    return [(np.asarray(c), np.asarray(t)) for d in items for c, t in zip(d["chunks"], d["targets"]) if len(t) > 0]
train_pairs, val_pairs = pairs_of(data["train"]), pairs_of(data["val"])
chunk_counts = np.array([len(d["chunks"]) for d in data["train"]])
target_share = np.mean([len(t) / max(len(c), 1) for c, t in train_pairs])
recalls = np.array(data["alignment_recall"])
print(f"train pairs {len(train_pairs):,} | validation pairs {len(val_pairs):,} | chunks per document median "
      f"{int(np.median(chunk_counts))}, p90 {int(np.percentile(chunk_counts, 90))} | target ≈ {100 * target_share:.0f}% of its chunk")
print(f"summary sentences aligned with word recall ≥ 0.5: {100 * np.mean(recalls >= 0.5):.1f}% (median {np.median(recalls):.2f}) | "
      f"chunks with no summary sentence: {100 * np.mean([len(t) == 0 for d in data['train'] for t in d['targets']]):.1f}%")

In [ ]:
# Cell 6 — The complete Phase-1 Architecture B (encoder–decoder), same parameter names as Phase 1, loaded strictly.
# Differences from the Phase-1 code are only additions: padding masks, and cached incremental decoding (Cell 10).
b_paths = sorted(glob.glob("/kaggle/input/**/architecture_B/model_B_state_dict.pt", recursive=True))
assert b_paths, "Add the dataset rohitbatra2004/arch-data-1 as an input."
B_ROOT = Path(b_paths[0]).parent
B_METADATA = json.load(open(B_ROOT / "model_B_metadata.json"))
B_CONFIG = B_METADATA["model_config"]
assert B_CONFIG["pre_norm"] and B_CONFIG["tie_embeddings"], "Phase-1 B is pre-LN with tied embeddings"

def gelu(x):
    return F.gelu(x, approximate="tanh")

def heads(t, n):
    B, T, Cd = t.shape
    return t.view(B, T, n, Cd // n).transpose(1, 2)

class SelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg["n_heads"]
        self.qkv = nn.Linear(cfg["d_model"], 3 * cfg["d_model"], bias=cfg["bias"])
        self.out_proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x, mask, cache=None):
        """mask: boolean, broadcastable to [B, 1, T, S]; cache: (k, v) of earlier decoder positions."""
        q, k, v = (heads(t, self.n_heads) for t in self.qkv(x).chunk(3, dim=-1))
        if cache is not None:
            k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        B, H, T, D = out.shape
        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * D)), (k, v)

class CrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg["n_heads"]
        d, b = cfg["d_model"], cfg["bias"]
        self.q_proj, self.k_proj = nn.Linear(d, d, bias=b), nn.Linear(d, d, bias=b)
        self.v_proj, self.out_proj = nn.Linear(d, d, bias=b), nn.Linear(d, d, bias=b)

    def memory_kv(self, memory):
        return heads(self.k_proj(memory), self.n_heads), heads(self.v_proj(memory), self.n_heads)

    def forward(self, x, kv, src_mask):
        out = F.scaled_dot_product_attention(heads(self.q_proj(x), self.n_heads), kv[0], kv[1], attn_mask=src_mask)
        B, H, T, D = out.shape
        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * D))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.fc2(gelu(self.fc1(x)))

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(cfg["d_model"], bias=cfg["bias"]), nn.LayerNorm(cfg["d_model"], bias=cfg["bias"])
        self.self_attn, self.ffn = SelfAttention(cfg), FeedForward(cfg)

    def forward(self, x, mask):
        x = x + self.self_attn(self.norm1(x), mask)[0]
        return x + self.ffn(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, b = cfg["d_model"], cfg["bias"]
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(d, bias=b), nn.LayerNorm(d, bias=b), nn.LayerNorm(d, bias=b)
        self.self_attn, self.cross_attn, self.ffn = SelfAttention(cfg), CrossAttention(cfg), FeedForward(cfg)

    def forward(self, x, self_mask, cross_kv, src_mask, cache=None):
        a, kv = self.self_attn(self.norm1(x), self_mask, cache)
        x = x + a
        x = x + self.cross_attn(self.norm2(x), cross_kv, src_mask)
        return x + self.ffn(self.norm3(x)), kv

class ArchitectureB(nn.Module):
    """Phase-1 encoder–decoder; dropout is not used here (frozen-model adaptation and generation)."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.d_model = d
        self.token_embedding = nn.Embedding(cfg["vocab_size"], d)
        self.encoder_position_embedding = nn.Embedding(cfg["src_context_length"], d)
        self.decoder_position_embedding = nn.Embedding(cfg["tgt_context_length"], d)
        self.encoder_layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg["n_encoder_layers"])])
        self.encoder_final_norm = nn.LayerNorm(d, bias=cfg["bias"])
        self.decoder_layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg["n_decoder_layers"])])
        self.decoder_final_norm = nn.LayerNorm(d, bias=cfg["bias"])
        self.output_bias = nn.Parameter(torch.zeros(cfg["vocab_size"]))

    def embed(self, ids, position_table, start=0):
        positions = torch.arange(start, start + ids.size(1), device=ids.device)
        return self.token_embedding(ids) * math.sqrt(self.d_model) + position_table(positions)[None]

    def encode(self, src):
        src_mask = src.ne(PAD_ID)[:, None, None, :]
        x = self.embed(src, self.encoder_position_embedding)
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return self.encoder_final_norm(x), src_mask

    def cross_kvs(self, memory):
        return [layer.cross_attn.memory_kv(memory) for layer in self.decoder_layers]

    def decode(self, dec, cross, src_mask, caches=None, start=0):
        T = dec.size(1)
        causal = torch.ones(T, start + T, dtype=torch.bool, device=dec.device).tril(start)[None, None]
        x = self.embed(dec, self.decoder_position_embedding, start)
        new = []
        for i, layer in enumerate(self.decoder_layers):
            x, kv = layer(x, causal, cross[i], src_mask, None if caches is None else caches[i])
            new.append(kv)
        return self.decoder_final_norm(x), new

    def logits(self, states):
        return F.linear(states, self.token_embedding.weight, self.output_bias)

saved = torch.load(b_paths[0], map_location="cpu", weights_only=True)
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved
check = ArchitectureB(B_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)                     # every Phase-1 tensor, including the decoder
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(B_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(B_CONFIG, indent=2))
print(f"Phase-1 Architecture B (encoder + decoder) verified from {B_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 7 — LoRA: y = W0·x + b + (α/r)·B(A(x)) on every linear projection of the encoder and decoder; only A and B train.
# B = 0 at start, so the adapted model initially equals Phase 1 (checked below on a short source/target pair).
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, base.out_features, bias=False)
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(x))

def make_lora_model():
    m = ArchitectureB(B_CONFIG)
    m.load_state_dict(PHASE1_STATE, strict=True)
    for p in m.parameters():
        p.requires_grad_(False)
    for layer in list(m.encoder_layers) + list(m.decoder_layers):
        modules = [layer.self_attn, layer.ffn] + ([layer.cross_attn] if hasattr(layer, "cross_attn") else [])
        for module in modules:
            for name, child in list(module.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(module, name, LoRALinear(child, LORA_RANK, LORA_ALPHA))
    return m

def full_forward(m, src, dec):
    memory, src_mask = m.encode(src)
    return m.logits(m.decode(dec, m.cross_kvs(memory), src_mask)[0])

probe_src, probe_dec = torch.randint(4, 1000, (2, 20)), torch.randint(4, 1000, (2, 7))
plain = ArchitectureB(B_CONFIG); plain.load_state_dict(PHASE1_STATE, strict=True); plain.eval()
adapted = make_lora_model().eval()
with torch.no_grad():
    assert torch.allclose(full_forward(plain, probe_src, probe_dec), full_forward(adapted, probe_src, probe_dec), atol=1e-4), \
        "LoRA with B = 0 must equal Phase 1"
lora_layers = [m for m in adapted.modules() if isinstance(m, LoRALinear)]
n_lora = sum(p.numel() for p in adapted.parameters() if p.requires_grad)
print(f"LoRA on {len(lora_layers)} projections — {n_lora:,} trainable parameters ({100 * n_lora / n_backbone:.2f}%)")
del plain, adapted

In [ ]:
# Cell 8 — Sequence-to-sequence summariser: encoder reads the chunk, decoder predicts the summary (loss on summary + [EOS]).
class SummaryEncDec(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, src, dec, labels):
        """src [B, S], dec [B, T] (= [BOS] + summary), labels [B, T] (= summary + [EOS], -100 on padding)."""
        memory, src_mask = self.backbone.encode(src)
        states, _ = self.backbone.decode(dec, self.backbone.cross_kvs(memory), src_mask)
        keep = labels.ne(-100)
        logits = self.backbone.logits(states[keep]).float()
        return F.cross_entropy(logits, labels[keep], reduction="sum").unsqueeze(0), keep.sum().float().unsqueeze(0)

torch.manual_seed(SEED)
model = SummaryEncDec(make_lora_model()).to(DEVICE)
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = list(TRAINABLE_NAMES)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
N_GPUS = torch.cuda.device_count()
train_model = nn.DataParallel(model) if N_GPUS > 1 else model
assert CHUNK_TOKENS + 2 <= B_CONFIG["src_context_length"] and MAX_TARGET_TOKENS + 1 <= B_CONFIG["tgt_context_length"]
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)} | "
      f"source ≤ {B_CONFIG['src_context_length']}, target ≤ {B_CONFIG['tgt_context_length']} tokens")

In [ ]:
# Cell 9 — Batches: pairs of similar length grouped together; encoder input [BOS] chunk [EOS], decoder input [BOS] summary,
# labels summary [EOS] (padding = -100).
def make_batch(pairs):
    srcs = [[BOS_ID] + c.tolist() + [EOS_ID] for c, _ in pairs]
    tgts = [t.tolist() for _, t in pairs]
    S, T = max(len(s) for s in srcs), max(len(t) for t in tgts) + 1
    src = torch.full((len(pairs), S), PAD_ID, dtype=torch.long)
    dec = torch.full((len(pairs), T), PAD_ID, dtype=torch.long)
    labels = torch.full((len(pairs), T), -100, dtype=torch.long)
    for b, (s, t) in enumerate(zip(srcs, tgts)):
        src[b, :len(s)] = torch.tensor(s)
        dec[b, :len(t) + 1] = torch.tensor([BOS_ID] + t)
        labels[b, :len(t) + 1] = torch.tensor(t + [EOS_ID])
    return src.to(DEVICE), dec.to(DEVICE), labels.to(DEVICE)

def length_batches(pairs, batch_size, seed):
    order = sorted(range(len(pairs)), key=lambda i: len(pairs[i][0]) + len(pairs[i][1]))
    batches = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    random.Random(seed).shuffle(batches)
    return batches

@torch.no_grad()
def validation_loss(net):
    net.eval()
    total, count = 0.0, 0.0
    for batch in length_batches(val_pairs, BATCH_SEQS * 2, SEED):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss_sum, n = net(*make_batch([val_pairs[i] for i in batch]))
        total, count = total + loss_sum.sum().item(), count + n.sum().item()
    return total / max(count, 1)

src, dec, labels = make_batch(train_pairs[:2])
print("Example batch: source", tuple(src.shape), "| decoder", tuple(dec.shape), "| target tokens:", int(labels.ne(-100).sum()),
      f"| {math.ceil(len(train_pairs) / BATCH_SEQS):,} batches per epoch")

In [ ]:
# Cell 10 — Cached greedy generation for the encoder–decoder: the encoder runs once per chunk, cross-attention keys/values
# are computed once, decoder self-attention keys/values are cached; repeated 4-grams blocked, min / max lengths per chunk.
def block_repeats(logits, generated, n):
    """Forbid tokens that would repeat an n-gram already generated in that row."""
    for b, seq in enumerate(generated):
        if len(seq) >= n - 1:
            prefix = tuple(seq[-(n - 1):])
            banned = [seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == prefix]
            if banned:
                logits[b, banned] = -1e4

@torch.no_grad()
def generate(bb, sources, max_new, min_new):
    """sources: encoder token lists → generated summary token lists (without [EOS])."""
    bb.eval()
    B, S = len(sources), max(len(s) for s in sources)
    src = torch.full((B, S), PAD_ID, dtype=torch.long, device=DEVICE)
    for b, s in enumerate(sources):
        src[b, :len(s)] = torch.tensor(s, device=DEVICE)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        memory, src_mask = bb.encode(src)
        cross = bb.cross_kvs(memory)
        token = torch.full((B, 1), BOS_ID, dtype=torch.long, device=DEVICE)
        caches, generated, done = None, [[] for _ in range(B)], [False] * B
        for step in range(max(max_new)):
            states, caches = bb.decode(token, cross, src_mask, caches, start=step)
            logits = bb.logits(states[:, -1]).float()
            logits[:, [PAD_ID, BOS_ID, UNK_ID]] = -1e4
            for b in range(B):
                if step < min_new[b]:
                    logits[b, EOS_ID] = -1e4
            block_repeats(logits, generated, NO_REPEAT_NGRAM)
            nxt = logits.argmax(-1)
            for b in range(B):
                if not done[b]:
                    if nxt[b].item() == EOS_ID or len(generated[b]) >= max_new[b]:
                        done[b] = True
                    else:
                        generated[b].append(nxt[b].item())
            if all(done):
                break
            token = nxt[:, None]
    return generated

def summarise(items, first_chunk_only=False):
    """Per document: list of per-chunk summaries (each a list of sentences)."""
    jobs = [(i, j, c) for i, d in enumerate(items) for j, c in enumerate(d["chunks"][:1] if first_chunk_only else d["chunks"])]
    jobs.sort(key=lambda job: len(job[2]))
    out = [dict() for _ in items]
    for s in range(0, len(jobs), GEN_BATCH):
        batch = jobs[s:s + GEN_BATCH]
        prompts = [[BOS_ID] + c.tolist() + [EOS_ID] for _, _, c in batch]
        max_new = [min(MAX_TARGET_TOKENS, int(MAX_NEW_FRACTION * len(c)) + 32) for _, _, c in batch]
        min_new = [int(MIN_NEW_FRACTION * len(c)) for _, _, c in batch]
        for (i, j, _), g in zip(batch, generate(model.backbone, prompts, max_new, min_new)):
            out[i][j] = split_sentences(tokenizer.decode(g).strip())
    return [[out[i][j] for j in sorted(out[i])] for i in range(len(items))]

def join_summary(chunk_summaries, keep_fraction=1.0):
    """Keep the first share of each chunk summary's sentences, drop sentences already used, join in document order."""
    seen, kept = set(), []
    for sentences in chunk_summaries:
        for s in sentences[:max(1, math.ceil(keep_fraction * len(sentences)))] if sentences else []:
            key = " ".join(WORD.findall(s.lower()))
            if key and key not in seen:
                seen.add(key); kept.append(s)
    return " ".join(kept)
print("Generation helpers ready.")

In [ ]:
# Cell 11 — ROUGE-1 / ROUGE-L F1 (word level, lower-cased) and repetition, for the held-out validation documents.
def rouge1_f(ref, hyp):
    overlap = sum((Counter(ref) & Counter(hyp)).values())
    return 0.0 if overlap == 0 else 2 * overlap / (len(ref) + len(hyp))

def rougeL_f(ref, hyp, cap=800):
    ref, hyp = ref[:cap], hyp[:cap]
    prev = [0] * (len(hyp) + 1)
    for x in ref:
        cur = [0]
        for j, y in enumerate(hyp, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return 0.0 if prev[-1] == 0 else 2 * prev[-1] / (len(ref) + len(hyp))

def score_summaries(refs, hyps):
    r1, rl, lengths, rep = [], [], [], []
    for ref, hyp in zip(refs, hyps):
        rw, hw = WORD.findall(ref.lower()), WORD.findall(hyp.lower())
        r1.append(rouge1_f(rw, hw)); rl.append(rougeL_f(rw, hw)); lengths.append(len(hw) / max(len(rw), 1))
        grams = list(zip(hw, hw[1:], hw[2:]))
        rep.append(1 - len(set(grams)) / max(len(grams), 1))
    return {"rouge1": float(np.mean(r1)), "rougeL": float(np.mean(rl)), "length_ratio": float(np.median(lengths)),
            "repeated_trigram_share": float(np.mean(rep))}

def describe(s):
    return (f"ROUGE-1 {s['rouge1']:.4f} | ROUGE-L {s['rougeL']:.4f} | length vs reference ×{s['length_ratio']:.2f} | "
            f"repeated trigrams {100 * s['repeated_trigram_share']:.1f}%")
print("Scoring ready (validation only).")

In [ ]:
# Cell 12 — Optimiser, learning-rate schedule and checkpoint resume (every 1,000 steps and every epoch).
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_pairs) / BATCH_SEQS)
total_updates = EPOCHS * math.ceil(steps_per_epoch / GRAD_ACCUM)
warmup_updates = int(WARMUP_FRACTION * total_updates)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda u: (u + 1) / max(warmup_updates, 1) if u < warmup_updates
                                              else max(0.0, (total_updates - u) / max(total_updates - warmup_updates, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def trainable_state():
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n in SAVED_NAMES}

LATEST, BEST = CKPT_DIR / "latest.pt", CKPT_DIR / "best.pt"
if not LATEST.exists():
    for previous in sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True)):
        if set(torch.load(previous, map_location="cpu", weights_only=False)["trainable"]) == set(SAVED_NAMES):
            shutil.copy(previous, LATEST)
            if Path(previous).with_name("best.pt").exists():
                shutil.copy(Path(previous).with_name("best.pt"), BEST)
            print("Copied checkpoints from", Path(previous).parent)
            break

state = {"epoch": 0, "step": 0, "best_val_loss": float("inf"), "best_epoch": 0, "history": [], "train_seconds": 0.0}
if LATEST.exists():
    ckpt = torch.load(LATEST, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"]); scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
    state = ckpt["state"]
    print(f"Resumed at epoch {state['epoch'] + 1}, step {state['step']}")
else:
    state["val_loss_phase1"] = validation_loss(train_model)        # epoch 0: the unadapted Phase-1 model
    print(f"No checkpoint found — training starts from the Phase-1 weights | validation loss before training "
          f"{state['val_loss_phase1']:.4f} (perplexity {math.exp(state['val_loss_phase1']):.1f})")
print(f"{steps_per_epoch:,} steps per epoch, {total_updates:,} optimiser updates in total")

In [ ]:
# Cell 13 — Training: next-token cross-entropy on the summary tokens, mixed precision, gradient accumulation, frozen
# backbone without dropout. Every epoch ends with the validation loss; best.pt keeps the epoch with the lowest one.
def save_latest():
    torch.save({"trainable": trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, LATEST)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
while state["epoch"] < EPOCHS:
    epoch = state["epoch"]
    batches = length_batches(train_pairs, BATCH_SEQS, SEED + epoch)
    train_model.train(); model.backbone.eval()
    start_time, loss_sum, token_sum = time.time(), 0.0, 0.0
    for step in range(state["step"], len(batches)):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            s, n = train_model(*make_batch([train_pairs[i] for i in batches[step]]))
        loss = s.sum() / n.sum().clamp(min=1)
        scaler.scale(loss / GRAD_ACCUM).backward()
        loss_sum, token_sum = loss_sum + s.sum().item(), token_sum + n.sum().item()
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(batches):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        state["step"] = step + 1
        if state["step"] % SAVE_EVERY_STEPS == 0:
            state["train_seconds"] += time.time() - start_time; start_time = time.time()
            save_latest()
            print(f"  epoch {epoch + 1} step {state['step']}/{len(batches)} | train loss {loss_sum / token_sum:.4f}", flush=True)
    state["train_seconds"] += time.time() - start_time
    val_loss = validation_loss(train_model)
    state["history"].append({"epoch": epoch + 1, "train_loss": loss_sum / max(token_sum, 1), "val_loss": val_loss})
    mark = ""
    if val_loss < state["best_val_loss"]:
        state.update(best_val_loss=val_loss, best_epoch=epoch + 1)
        torch.save({"trainable": trainable_state(), "epoch": epoch + 1, "val_loss": val_loss}, BEST)
        mark = "  <- best"
    state["epoch"], state["step"] = epoch + 1, 0
    save_latest()
    print(f"Epoch {epoch + 1} | train loss {loss_sum / max(token_sum, 1):.4f} | validation loss {val_loss:.4f} "
          f"(perplexity {math.exp(val_loss):.1f}){mark}", flush=True)
print(f"Best epoch {state['best_epoch']} (validation loss {state['best_val_loss']:.4f}) | training time {state['train_seconds'] / 60:.1f} min")

In [ ]:
# Cell 14 — Validation summaries with the best model: all chunks vs first chunk only (long-document comparison), and the
# share of each chunk summary to keep (length control). The best keep-fraction is used for the test summaries.
best = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
refs = [d["summary"] for d in data["val"]]
start_time = time.time()
val_chunk_summaries = summarise(data["val"])
gen_minutes = (time.time() - start_time) / 60
results = {}
for keep in KEEP_FRACTIONS:
    results[f"all_chunks_keep_{keep}"] = score_summaries(refs, [join_summary(c, keep) for c in val_chunk_summaries])
results["first_chunk_only"] = score_summaries(refs, [join_summary(c) for c in summarise(data["val"], first_chunk_only=True)])
for k, v in results.items():
    print(f"  {k:24s}: {describe(v)}")
KEEP = max(KEEP_FRACTIONS, key=lambda k: results[f"all_chunks_keep_{k}"]["rouge1"] + results[f"all_chunks_keep_{k}"]["rougeL"])
print(f"Chosen keep-fraction {KEEP} | validation generation {gen_minutes:.1f} min for {len(data['val'])} documents")
example = data["val"][0]
print("\nExample (validation) — reference:\n", example["summary"][:700], "\n\nGenerated:\n", join_summary(val_chunk_summaries[0], KEEP)[:700])
json.dump({"best_epoch": best["epoch"], "history": state["history"], "val_loss_phase1": state.get("val_loss_phase1"),
           "validation": results, "keep_fraction": KEEP}, open(OUT_DIR / "val_metrics.json", "w"), indent=2)

In [ ]:
# Cell 15 — Test summaries (all chunks, chosen keep-fraction), format checks and submission.zip.
# prediction.json is a JSON list of {"id": ..., "summary": "..."} with exactly one non-empty summary per test document.
start_time = time.time()
test_chunk_summaries = summarise(data["test"])
test_minutes = (time.time() - start_time) / 60
prediction = [{"id": d["id"], "summary": join_summary(c, KEEP)} for d, c in zip(data["test"], test_chunk_summaries)]
for p, d in zip(prediction, data["test"]):
    if not p["summary"].strip():                                  # never submit an empty summary
        p["summary"] = tokenizer.decode(d["chunks"][0][:200].tolist())
assert [p["id"] for p in prediction] == [d["id"] for d in test_docs] and len({p["id"] for p in prediction}) == len(prediction)
assert all(isinstance(p["summary"], str) and p["summary"].strip() for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=1)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
words = np.array([len(p["summary"].split()) for p in prediction])
print(f"submission.zip: {len(prediction)} summaries | words per summary median {int(np.median(words))} "
      f"(training references median {int(np.median([len(d['summary'].split()) for d in train_docs]))}) | generation {test_minutes:.1f} min")

In [ ]:
# Cell 16 — Run summary for the README, the experiment log and results/submissions.csv.
summary = {
    "run": RUN_NAME, "backbone": "Architecture B — encoder–decoder Transformer (Phase 1), encoder and decoder",
    "technique": "LoRA on all encoder/decoder projections, seq2seq chunk-level summarization, joined with redundancy removal",
    "lora_rank": LORA_RANK, "trainable_params": n_trainable, "total_params": n_total,
    "train_pairs": len(train_pairs), "val_docs": len(data["val"]), "epochs": state["epoch"], "best_epoch": state["best_epoch"],
    "val_loss_phase1": state.get("val_loss_phase1"), "val_loss_best": state["best_val_loss"],
    "train_minutes": round(state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "validation": results, "keep_fraction": KEEP, "test_generation_minutes": round(test_minutes, 1),
    "test_words_median": int(np.median(words)),
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))